To run this please download openmoji-618x618-color.zip from 

https://github.com/hfg-gmuend/openmoji/releases/tag/16.0.0

In [58]:
import pandas as pd
import zipfile
import os
from torchvision import transforms
from PIL import Image, ImageEnhance
import os
import torch
from torch.utils.data import Dataset, DataLoader

In [18]:
df = pd.read_csv("data//openmoji.csv")
df.head()

,emoji,hexcode,group,subgroups,annotation,tags,openmoji_tags,openmoji_author,openmoji_date,skintone,skintone_combination,skintone_base_emoji,skintone_base_hexcode,unicode,order
0,😀,1F600,smileys-emotion,face-smiling,grinning face,"cheerful, cheery, face, grin, grinning, happy,...","smile, happy",Emily Jäger,2018-04-18,NaN,NaN,NaN,NaN,1,1.0
1,😃,1F603,smileys-emotion,face-smiling,grinning face with big eyes,"awesome, big, eyes, face, grin, grinning, happ...","eyes, teeth",Emily Jäger,2018-04-18,NaN,NaN,NaN,NaN,0.6,2.0
2,😄,1F604,smileys-emotion,face-smiling,grinning face with smiling eyes,"eye, eyes, face, grin, grinning, happy, laugh,...","happy, teeth",Emily Jäger,2018-04-18,NaN,NaN,NaN,NaN,0.6,3.0
3,😁,1F601,smileys-emotion,face-smiling,beaming face with smiling eyes,"beaming, eye, eyes, face, grin, grinning, happ...","happy, teeth, mouth",Emily Jäger,2018-04-18,NaN,NaN,NaN,NaN,0.6,4.0
4,😆,1F606,smileys-emotion,face-smiling,grinning squinting face,"closed, eyes, face, grinning, haha, hahaha, ha...",NaN,Emily Jäger,2018-04-18,NaN,NaN,NaN,NaN,0.6,5.0


In [20]:
df['group'].value_counts()

group
people-body        2261
extras-openmoji     381
flags               270
objects             264
symbols             224
travel-places       218
smileys-emotion     169
animals-nature      159
food-drink          131
extras-unicode      121
activities           85
component             9
Name: count, dtype: int64

In [21]:
smileys = df[df['group'] == "smileys-emotion"]

In [25]:
smileys['subgroups'].value_counts()

subgroups
face-concerned            26
heart                     25
face-neutral-skeptical    16
face-smiling              14
emotion                   14
face-unwell               12
face-affection             9
cat-face                   9
face-negative              8
face-costume               8
face-hand                  7
face-tongue                6
face-sleepy                6
face-hat                   3
face-glasses               3
monkey-face                3
Name: count, dtype: int64

In [42]:
# Filter out the specified subgroups
excluded_groups = ["heart", "cat-face", "emotion", "face-costume", "monkey-face", "face-hat", "face-glasses", "face-hand"]
excluded_not_faces = ["1F480", "1F636-200D-1F32B-FE0F", "1F643", "2620"]
smileys = smileys[~smileys['subgroups'].isin(excluded_groups)]
smileys = smileys[~smileys['hexcode'].isin(excluded_not_faces)]

In [45]:
smileys['subgroups'].value_counts()
print(f"Total: {smileys['subgroups'].value_counts().sum()}")

Total: 93


In [55]:
zip_path = 'data/openmoji-618x618-color.zip'
output_dir = 'data/images'
os.makedirs(output_dir, exist_ok=True)

valid_hexcodes = set(smileys['hexcode'].astype(str))

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    all_files = zip_ref.namelist()
    
    extracted_count = 0
    for file_name in all_files:
        base_name = os.path.basename(file_name).split('.')[0]
        
        if base_name in valid_hexcodes:
            zip_ref.extract(file_name, output_dir)
            extracted_count += 1

print(f"Successfully extracted {extracted_count} images to {output_dir}")

Successfully extracted 93 images to data/images


In [ ]:
def augment_image(image_path, output_dir):
    """
    Augment a single image with 5 variations:
    _1: horizontal flip
    _2: rotate 15 degrees
    _3: rotate -15 degrees
    _4: increase brightness
    _5: decrease brightness
    """
    # Load image
    img = Image.open(image_path).convert('RGBA')
    
    # Get filename without extension
    filename = os.path.basename(image_path)
    name_without_ext = os.path.splitext(filename)[0]
    
    # 1. Horizontal flip
    img_flip = img.transpose(Image.FLIP_LEFT_RIGHT)
    img_flip.save(os.path.join(output_dir, f"{name_without_ext}_1.png"), 'PNG')
    
    # 2. Rotate 15 degrees (expand=True keeps the whole image, fillcolor for transparency)
    img_rot15 = img.rotate(15, expand=True, fillcolor=(0, 0, 0, 0))
    img_rot15.save(os.path.join(output_dir, f"{name_without_ext}_2.png"), 'PNG')
    
    # 3. Rotate -15 degrees
    img_rot_neg15 = img.rotate(-15, expand=True, fillcolor=(0, 0, 0, 0))
    img_rot_neg15.save(os.path.join(output_dir, f"{name_without_ext}_3.png"), 'PNG')
    
    # 4. Increase brightness (1.3x)
    enhancer_bright = ImageEnhance.Brightness(img)
    img_bright = enhancer_bright.enhance(1.3)
    img_bright.save(os.path.join(output_dir, f"{name_without_ext}_4.png"), 'PNG')
    
    # 5. Decrease brightness (0.7x)
    enhancer_dark = ImageEnhance.Brightness(img)
    img_dark = enhancer_dark.enhance(0.7)
    img_dark.save(os.path.join(output_dir, f"{name_without_ext}_5.png"), 'PNG')
    
    print(f"✓ Augmented {filename}")


def augment_all_images(input_dir, output_dir=None):
    """
    Augment all PNG images in a directory.
    If output_dir is None, saves to the same directory as input.
    """
    # Use same directory if output not specified
    if output_dir is None:
        output_dir = input_dir
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Get all PNG files
    png_files = [f for f in os.listdir(input_dir) 
                 if f.endswith('.png') and not any(f.endswith(f'_{i}.png') for i in range(1, 6))]
    
    print(f"Found {len(png_files)} original PNG files to augment")
    print(f"Output directory: {output_dir}\n")
    
    # Process each image
    for png_file in png_files:
        image_path = os.path.join(input_dir, png_file)
        augment_image(image_path, output_dir)
    
    print(f"\n✅ Done! Created {len(png_files) * 5} augmented images")

input_directory = "data/images"

augment_all_images(input_directory)

Found 93 original PNG files to augment
Output directory: data/images

✓ Augmented 1FAE8.png
✓ Augmented 1F611.png
✓ Augmented 1F92F.png
✓ Augmented 1F631.png
✓ Augmented 1F600.png
✓ Augmented 1F979.png
✓ Augmented 1F61C.png
✓ Augmented 1F633.png
✓ Augmented 1F641.png
✓ Augmented 1F915.png
✓ Augmented 1F629.png
✓ Augmented 1F925.png
✓ Augmented 1F60B.png
✓ Augmented 1F644.png
✓ Augmented 1F62B.png
✓ Augmented 1F924.png
✓ Augmented 1F614.png
✓ Augmented 1F61F.png
✓ Augmented 1FAE5.png
✓ Augmented 1F642.png
✓ Augmented 1F923.png
✓ Augmented 1F601.png
✓ Augmented 1F92C.png
✓ Augmented 1F642-200D-2195-FE0F.png
✓ Augmented 1F62D.png
✓ Augmented 1F635-200D-1F4AB.png
✓ Augmented 1F637.png
✓ Augmented 1F609.png
✓ Augmented 1F615.png
✓ Augmented 263A.png
✓ Augmented 1F634.png
✓ Augmented 1F622.png
✓ Augmented 1F62A.png
✓ Augmented 1F928.png
✓ Augmented 1F603.png
✓ Augmented 1F975.png
✓ Augmented 1F62F.png
✓ Augmented 1F617.png
✓ Augmented 1F602.png
✓ Augmented 1F974.png
✓ Augmented 1F627.png
✓ A